####Saving the external locations in variables

In [0]:
checkpoint = spark.sql("DESCRIBE EXTERNAL LOCATION `checkpoints`").select('url').collect()[0][0]
landing = spark.sql("DESCRIBE EXTERNAL LOCATION `landing`").select('url').collect()[0][0]

In [0]:
dbutils.widgets.text(name='env',defaultValue='',label='Enter the Environment')
env = dbutils.widgets.get('env')

#### Creating the autoloader readstream

In [0]:
myschema = "Record_ID int,Count_point_id int,Direction_of_travel string,Year int,Count_date string,hour int,Region_id int,Region_name string,Local_authority_name string,Road_name string,Road_category_ID int,Start_junction_road_name string,End_junction_road_name string,Latitude DOUBLE,Longitude DOUBLE,Link_length_km DOUBLE,Pedal_cycles INT,Two_wheeled_motor_vehicles INT,Cars_and_taxis INT,Buses_and_coaches INT,LGV_type INT,HGV_type INT,EV_car INT,EV_bike INT"

In [0]:
def raw_traffic_load():
    from pyspark.sql.functions import current_timestamp

    raw_traffic_load = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation",f'{checkpoint}/rawTrafficLoad/schema')\
        .schema(myschema)\
            .option('header','true')\
                .load(f"{landing}/raw_traffic")\
                .withColumn('Extract_time', current_timestamp())
    print('Bronze table read success')
    print('****************************')                  

    return raw_traffic_load            

####Creating incremental writeStream with availableNow trigger

In [0]:
def write_traffic_bronze(streamingDF,environment):
    streamingDF.writeStream \
        .format("delta") \
        .option("checkpointLocation", f"{checkpoint}/rawTrafficLoad/checkpnt")\
            .outputMode('append')\
            .trigger(availableNow = True)\
                .toTable(f"`{environment}catalog`.`bronze`.`raw_traffic`")

    print('Bronze table load success')
    print('****************************')            

####Calling all the functions

In [0]:
readDf = raw_traffic_load()
write_traffic_bronze(readDf,env)